# Transformer Summarization - Experiments Lab

This notebook is designed to help you run experiments, tune hyperparameters, and reach your goal of 70% accuracy on the CNN/DailyMail dataset.

## 1. Setup Environment
Make sure you are running on a **GPU Runtime**.

In [ ]:
import tensorflow as tf
print("Num GPUs Available: ", len(tf.config.list_physical_devices('GPU')))
if tf.config.list_physical_devices('GPU'):
    print("GPU is available. Ready to train!")
else:
    print("WARNING: GPU not available. Go to Runtime > Change runtime type > T4 GPU")

In [ ]:
# Clone the repository
!git clone https://github.com/AbdulRasheed1011/Transformer_from_scratch.git
%cd Transformer_from_scratch

# Install dependencies (including rouge-score for evaluation)
!pip install -r requirements.txt

## 2. Prepare Data
Download the dataset and train the tokenizer. This only needs to be done once.

In [ ]:
!python -m src.data_ingestion.preprocessing

## 3. Configure Experiment
Use the function below to easily change hyperparameters (Model Depth, Dimensions, Epochs, Batch Size) without editing files manually.

**Goal:** Reach 70% Test Accuracy.

In [ ]:
import yaml
import os

def update_config(epochs=20, batch_size=32, num_layers=4, d_model=128, dff=512, num_heads=8):
    # Try finding config robustly
    if os.path.exists("config/config.yaml"):
        config_path = "config/config.yaml"
    elif os.path.exists("../config/config.yaml"):
        config_path = "../config/config.yaml"
    elif os.path.exists("/content/Transformer_from_scratch/config/config.yaml"):
        config_path = "/content/Transformer_from_scratch/config/config.yaml"
    else:
         raise FileNotFoundError("Could not find config/config.yaml. Make sure you are in the project root or run the setup cell.")

    with open(config_path, "r") as f:
        config = yaml.safe_load(f)
    
    # Update parameters
    config['training']['epochs'] = epochs
    config['training']['batch_size'] = batch_size
    config['model']['num_layers'] = num_layers
    config['model']['d_model'] = d_model
    config['model']['dff'] = dff
    config['model']['num_heads'] = num_heads

    with open(config_path, "w") as f:
        yaml.dump(config, f)
    
    print(f"Configuration updated!\nEpochs: {epochs}, Batch Size: {batch_size}, Layers: {num_layers}, d_model: {d_model}")

# EXAMPLE: Reset to Baseline
update_config(epochs=10, batch_size=64, num_layers=4, d_model=128, dff=512)


## 4. Run Training
Run the training script. It will output **Validation Accuracy** and **ROUGE Scores**.

**Note:** Record the final Test Accuracy and Training Time in your `Readme.md` Experiments Log.

In [ ]:
# Run Experiment 1: Baseline
# update_config(epochs=10, batch_size=64, num_layers=4, d_model=128, dff=512)
!python -m src.training.train

In [ ]:
# Run Experiment 2: Increased Capacity
# update_config(epochs=20, batch_size=32, num_layers=6, d_model=256, dff=1024)
# !python -m src.training.train

## 5. Inference / Demo
Test the trained model on a custom sentence.

In [ ]:
from src.inference.inference import Inference

# Initialize inference (loads latest checkpoint)
inference = Inference()

text = "The transformer model is a deep learning architecture that relies on the self-attention mechanism, weighing the significance of each part of the input data. It is used primarily in the field of natural language processing (NLP)."
summary = inference.predict(text)

print("Original:", text)
print("Summary:", summary)